In [ ]:
%cd c:\Users\almei\Documents\GitHub\precision-data
import csv
import pandas as pd
from pathlib import Path

def read_csv(path: str | Path, encoding: str = "utf-8") -> pd.DataFrame:
    path = Path(path)

    with path.open("r", encoding=encoding, newline="") as f:
        lines = [ln for ln in f.read().splitlines() if ln.strip()]

    # Header is normal CSV
    header = next(csv.reader([lines[0]], delimiter=",", quotechar='"'))
    ncols = len(header)

    rows = []
    for ln in lines[1:]:
        s = ln.strip()

        # If the whole line is quoted, unwrap it and unescape doubled quotes
        if s.startswith('"') and s.endswith('"'):
            s = s[1:-1].replace('""', '"')

        # Parse the (possibly unwrapped) line as CSV
        row = next(csv.reader([s], delimiter=",", quotechar='"'))

        # If we ended up with extra fields, merge them back into the FIRST column
        # until the row matches the header width.
        while len(row) > ncols:
            row[0] = f"{row[0]},{row[1]}"
            del row[1]

        # Optional: if it ends up short, you can pad (or raise)
        if len(row) != ncols:
            raise ValueError(f"Malformed row with {len(row)} fields (expected {ncols}): {ln}")

        rows.append(row)

    return pd.DataFrame(rows, columns=header)

# Example usage:
df = read_csv('vocabularies/SOURCE_TO_CONCEPT_MAP.csv')
print(df.head(20))


In [ ]:
%cd c:\Users\almei\Documents\GitHub\precision-data
from pathlib import Path
from aatm.aio.translators import AsyncGeminiTranslator
from aatm.terminology_mapper import TerminologyMapper
from aatm.retrievers import CHROMADB_RETRIEVER_MODEL_REGISTRY as model_registry
from aatm.retrievers import load_chromadb_retriever
from aatm.selectors import FirstResultSelector
from aatm.rerankers import Qwen3Reranker, BM25Reranker

model_name = "text-embedding-3-large"

translator = AsyncGeminiTranslator(model="gemini-2.5-flash")
retriever = load_chromadb_retriever(model_name)
selector = FirstResultSelector()

tm = TerminologyMapper(
    translator=translator,
    retriever=retriever,
    selector=selector, 
    batch_size=10,
    rate_limit=model_registry[model_name].get("rate_limit"),
    output_dir=Path(model_registry[model_name]["output_path"]),
)

mapped_concepts_df = await tm.amap(
    'vocabularies/SOURCE_TO_CONCEPT_MAP.csv',
)